# Dataset

In [52]:
import pandas as pd

df = pd.read_csv("insurance.csv")

X = df.drop("charges", axis=1)
y = df["charges"]

# Conversion

In [53]:
import numpy as np

categorical_cols = ["sex", "smoker", "region"]
numerical_cols = ["age", "bmi", "children"]

encoder = OneHotEncoder(sparse_output=False)
X_num = X[numerical_cols].values
X_cat = encoder.fit_transform(X[categorical_cols])

X_processed = np.hstack([X_num, X_cat])

# Split

In [54]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import torch

X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y.values.reshape(-1, 1), test_size=0.2, random_state=42
)

# Escalar features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Escalar target
y_scaler = StandardScaler()
y_train = y_scaler.fit_transform(y_train)
y_test = y_scaler.transform(y_test)

# Convertir a tensores
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

n_samples, n_features = X_train.shape

# Model

In [55]:
import torch.nn as nn

class LinearRegression(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(LinearRegression, self).__init__()
        self.lin = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.lin(x)

model = LinearRegression(n_features, 1)

# Train

In [56]:
learning_rate = 0.01
n_iters = 500

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(n_iters):
    y_pred = model(X_train)
    loss = loss_fn(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if (epoch+1) % 50 == 0:
        print(f"epoch {epoch+1}, loss = {loss.item():.2f}")

epoch 50, loss = 0.35
epoch 100, loss = 0.27
epoch 150, loss = 0.26
epoch 200, loss = 0.26
epoch 250, loss = 0.26
epoch 300, loss = 0.26
epoch 350, loss = 0.26
epoch 400, loss = 0.26
epoch 450, loss = 0.26
epoch 500, loss = 0.26


# Eval

In [57]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
with torch.no_grad():
    test_pred_scaled = model(X_test)

# Desescalar
predictions = y_scaler.inverse_transform(test_pred_scaled.numpy())
actual = y_scaler.inverse_transform(y_test.numpy())

# Evaluar métricas
mae = mean_absolute_error(actual, predictions)
mse = mean_squared_error(actual, predictions)
rmse = np.sqrt(mse)


print("MAE:", mae)
print("RMSE:", rmse)

MAE: 4181.232421875
RMSE: 5796.288467631679


# Report

In [58]:
report = pd.DataFrame({
    "Actual Charges": actual.flatten(),
    "Predicted Charges": predictions.flatten(),
    "Error": actual.flatten() - predictions.flatten(),
})

report.to_csv("predictions_report.csv", index=False)

print("\nReporte generado: predictions_report.csv")



Reporte generado: predictions_report.csv


# Example

In [59]:
print("\nPrimeras predicciones:")
print(report.head())


Primeras predicciones:
   Actual Charges  Predicted Charges        Error
0     9095.068359        8969.750000   125.318359
1     5272.176270        7068.660645 -1796.484375
2    29330.982422       36857.625000 -7526.642578
3     9301.893555        9455.082031  -153.188477
4    33750.292969       26973.333984  6776.958984
